# Oriented Bounding Box (OBB) vs Horizontal Bounding Box (HBB) Comparison

This notebook provides a comprehensive comparison between Oriented Bounding Boxes (OBB) and Horizontal Bounding Boxes (HBB) for object detection using YOLO models.

## Overview

This experiment evaluates the performance of different YOLO architectures (v8, v9, v10, v11, v12) with both OBB and HBB detection heads. The comparison includes:

1. **Model Training** - Training YOLO models with OBB and HBB configurations
2. **Evaluation Metrics** - mAP, Precision, Recall analysis
3. **Model Inference** - Running predictions on test data

## Requirements

```bash
pip install ultralytics torch torchvision numpy pandas opencv-python matplotlib seaborn
```

---
## 1. Setup and Imports

In [ ]:
import os
import torch
import numpy as np
import cv2
import pandas as pd
from ultralytics import YOLO
from pathlib import Path

# Check GPU availability
print(f"CUDA Available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)}")

In [ ]:
# Optional: Configure Ultralytics datasets directory
# from ultralytics import settings
# settings.update({"datasets_dir": "/path/to/your/datasets"})

---
## 2. Data Preprocessing Utilities

Utility functions for handling OBB label format and data validation.

In [ ]:
def fix_yolo_obb_coordinates(label_path):
    """Fixes potential issues in YOLO OBB format label files.

    Args:
        label_path (str): Path to the label file.
    """
    with open(label_path, "r") as f:
        lines = f.readlines()

    fixed_lines = []
    for line in lines:
        parts = line.strip().split()
        class_id = int(parts[0])
        coordinates = [float(coord) for coord in parts[1:]]

        # Check if coordinates are within 0-1 range:
        for i, coord in enumerate(coordinates):
            if coord < 0 or coord > 1:
                print(f"Warning: Coordinate {coord} in {label_path} is out of range (0-1). Clipping to [0, 1].")
                coordinates[i] = max(0, min(coord, 1))

        fixed_line = f"{class_id} {' '.join(map(str, coordinates))}\n"
        fixed_lines.append(fixed_line)

    with open(label_path, "w") as f:
        f.writelines(fixed_lines)


def validate_dataset_labels(dataset_dir):
    """Validate and fix OBB labels in a dataset directory.
    
    Args:
        dataset_dir (str): Path to dataset directory containing 'labels' folder.
    """
    labels_dir = os.path.join(dataset_dir, "labels")
    if not os.path.exists(labels_dir):
        print(f"Labels directory not found: {labels_dir}")
        return
    
    for filename in os.listdir(labels_dir):
        if filename.endswith(".txt"):
            label_path = os.path.join(labels_dir, filename)
            fix_yolo_obb_coordinates(label_path)
            print(f"Processed: {filename}")

In [ ]:
# Example: Validate dataset labels (uncomment to run)
# validate_dataset_labels("datasets/your_dataset/valid")

---
## 3. Model Training

### 3.1 HBB Model Training

Training standard Horizontal Bounding Box detection models.

In [ ]:
# HBB Detection Models
# Uncomment the model you want to train

# model = YOLO('yolov8x.pt')  
# model = YOLO('yolov9e.pt')  
# model = YOLO('yolov10x.pt')  
# model = YOLO('yolo11x.pt')  
# model = YOLO('yolo12x.pt')  

# Train HBB model
# results = model.train(data='data.yaml', imgsz=1024, batch=8)

### 3.2 OBB Model Training

Training Oriented Bounding Box detection models.

In [ ]:
# OBB Detection Models
# Uncomment the model you want to train

# YOLOv8-OBB
# model = YOLO('yolov8x-obb.pt')

# YOLOv9-OBB
# model = YOLO('yolov9-obb.yaml')
# model.load("yolov9e.pt") 

# YOLOv10-OBB
# model = YOLO('yolov10-obb.yaml') 
# model.load("yolov10x.pt") 

# YOLOv11-OBB
# model = YOLO('yolo11x-obb.pt')

# YOLOv12-OBB (Primary model used in experiments)
model = YOLO('yolo12x-obb.yaml') 
model.load('yolo12x.pt')

# Train OBB model
results = model.train(data='data.yaml', imgsz=1024, batch=8)

In [ ]:
# Resume training from checkpoint (if needed)
# model = YOLO("runs/obb/train/weights/last.pt")
# results = model.train(resume=True)

---
## 4. Model Evaluation

Evaluate trained models on test set and report metrics.

In [ ]:
def evaluate_model(model_path, data_yaml="data.yaml", split="test", imgsz=1024):
    """Evaluate a YOLO model and print detailed metrics.
    
    Args:
        model_path (str): Path to model weights.
        data_yaml (str): Path to data configuration file.
        split (str): Dataset split to evaluate ('train', 'val', 'test').
        imgsz (int): Image size for evaluation.
        
    Returns:
        dict: Dictionary containing all evaluation metrics.
    """
    model = YOLO(model_path)
    metrics = model.val(
        data=data_yaml, 
        split=split, 
        save_json=True, 
        save_txt=True, 
        imgsz=imgsz, 
        plots=True, 
        batch=16, 
        single_cls=True, 
        save_conf=True
    )

    # Extract metrics
    results = {
        'mAP50_95': metrics.box.map,
        'mAP50': metrics.box.map50,
        'mAP75': metrics.box.map75,
        'precision': metrics.box.p[0],
        'recall': metrics.box.r[0],
        'maps_per_class': metrics.box.maps
    }

    # Print results
    print("\n" + "="*50)
    print("Model Evaluation Results")
    print("="*50)
    print(f"mAP@.5:.95:   {results['mAP50_95']:.3f}")
    print(f"mAP@.5:       {results['mAP50']:.3f}")
    print(f"mAP@.75:      {results['mAP75']:.3f}")
    print(f"Precision:    {results['precision']:.3f}")
    print(f"Recall:       {results['recall']:.3f}")
    print("="*50)
    
    return results

In [ ]:
# Evaluate model
# results = evaluate_model('runs/obb/train/weights/best.pt', split='test')

---
## 5. Model Inference

Run inference on test images.

In [ ]:
def run_inference(model_path: str, source_dir: str, conf: float = 0.25, save: bool = True):
    """Run model inference on a directory of images.
    
    Args:
        model_path: Path to model weights.
        source_dir: Directory containing images.
        conf: Confidence threshold.
        save: Whether to save predictions.
    """
    model = YOLO(model_path)
    results = model.predict(
        source=source_dir,
        show_labels=False,
        show_conf=False,
        conf=conf,
        save=save,
        imgsz=[1024, 1024]
    )
    print("Inference complete!")
    return results

In [ ]:
# Example inference
# results = run_inference(
#     model_path='runs/obb/train/weights/best.pt',
#     source_dir='datasets/test/images',
#     conf=0.25,
#     save=True
# )

---
## 6. Results Summary

Template for recording experimental results.

In [ ]:
# Create results summary table
results_template = pd.DataFrame({
    'Model': ['YOLOv8-HBB', 'YOLOv8-OBB', 'YOLOv12-HBB', 'YOLOv12-OBB'],
    'mAP@.5': [0.0, 0.0, 0.0, 0.0],
    'mAP@.5:.95': [0.0, 0.0, 0.0, 0.0],
    'Precision': [0.0, 0.0, 0.0, 0.0],
    'Recall': [0.0, 0.0, 0.0, 0.0],
    'Training Time (h)': [0.0, 0.0, 0.0, 0.0]
})

print("Results Template:")
print(results_template.to_markdown(index=False))